<a href="https://colab.research.google.com/github/mstya/yolo-candy-detector/blob/main/train-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Installing required libs**

In [1]:
!pip install ultralytics
!pip install gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 3.2 MB/s eta 0:00:00


**Verify NVIDIA GPU Availability**

In [2]:
!nvidia-smi

Mon Aug 31 14:23:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Downloading a dataset from GoogleDrive**

In [9]:
!gdown 1tRvPHNL5lJPJGoBw0_PmMp_y_-t6-6af
!unzip -o -q 'data.zip'

Downloading...
From (original): https://drive.google.com/uc?id=1tRvPHNL5lJPJGoBw0_PmMp_y_-t6-6af
From (redirected): https://drive.google.com/uc?id=1tRvPHNL5lJPJGoBw0_PmMp_y_-t6-6af&confirm=t&uuid=c241a59b-aa74-4478-bb3d-c19413e2f2ba
To: /content/data.zip
100% 423M/423M [00:02<00:00, 157MB/s]


**Split images into train and validation folders**

In [10]:
!curl -o /content/train_val_split.py https://raw.githubusercontent.com/mstya/yolo-candy-detector/refs/heads/main/src/train_val_split.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3382  100  3382    0     0  16294      0 --:--:-- --:--:-- --:--:-- 16338


In [12]:
!python /content/train_val_split.py --targetpath="/content/split-data" --datapath="/content/data" --train_pct=0.9

Created folder at /content/split-data/train/images.
Created folder at /content/split-data/train/labels.
Created folder at /content/split-data/validation/images.
Created folder at /content/split-data/validation/labels.
Number of image files: 101
Number of annotation files: 101
Images moving to train: 90
Images moving to validation: 11


**Configure Training**

In [15]:
# Python function to automatically create data.yaml config file
# 1. Reads "classes.txt" file to get list of class names
# 2. Creates data dictionary with correct paths to folders, number of classes, and names of classes
# 3. Writes data in YAML format to data.yaml

import yaml
import os

root = '/content'

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

  # Read class.txt to get class names
  if not os.path.exists(path_to_classes_txt):
    print(f'classes.txt file not found! Please create a classes.txt labelmap and move it to {path_to_classes_txt}')
    return
  with open(path_to_classes_txt, 'r') as f:
    classes = []
    for line in f.readlines():
      if len(line.strip()) == 0: continue
      classes.append(line.strip())
  number_of_classes = len(classes)

  # Create data dictionary
  data = {
      'path': f'{root}/data',
      'train': f'{root}/split-data/train/images',
      'val': f'{root}/split-data/validation/images',
      'nc': number_of_classes,
      'names': classes
  }

  # Write data to YAML file
  with open(path_to_data_yaml, 'w') as f:
    yaml.dump(data, f, sort_keys=False)
  print(f'Created config file at {path_to_data_yaml}')

  return

# Define path to classes.txt and run function
path_to_classes_txt = f'{root}/data/classes.txt'
path_to_data_yaml = f'{root}/data.yaml'

create_data_yaml(path_to_classes_txt, path_to_data_yaml)

Created config file at /content/data.yaml


**Training**

In [ ]:
!yolo detect train data=/content/data.yaml model=yolo11s.pt epochs=60 imgsz=640 name=candy-detector

**Test model**

In [ ]:
!yolo detect predict model=runs/detect/candy-detector/weights/best.pt source=/content/split-data/validation/images save=True name=candy-detector-predict

In [ ]:
import glob
from IPython.display import Image, display
for image_path in glob.glob(f'runs/detect/candy-detector-predict/*.jpg')[:10]:
  display(Image(filename=image_path, height=400))
  print('\n')

In [ ]:
!wget -O /content/yolo_detect.py https://raw.githubusercontent.com/mstya/yolo-candy-detector/refs/heads/main/src/yolo_detect.py

**Downloading test video**

In [ ]:
!gdown 11i8G4R9KPm3GJsWqhzcAbys7W3YkE_XI
!python /content/yolo_detect.py --model runs/detect/candy-detector/weights/best.pt --source /content/video.mp4 --resolution 1280x720